# Bomberland MAPPO — Colab Training Pipeline

**Runtime:** GPU (T4) recommended for training. CPU works for eval/export.

**Current best checkpoint:** `checkpoint_update_1700.pth` (~36% win)

**Pipeline:**
1. Setup + mount Google Drive
2. Eval all checkpoints → pick best
3. Train from `checkpoint_update_1700` with `--no-dense-anneal`
4. Eval again → export `submission.zip`

In [ ]:
# Cell 1 — Clone repo (re-run safe)
GITHUB_REPO = "https://github.com/PandoraMiracle/Bomberland_GDGoC.git"

!rm -rf /content/bomberland
!git clone {GITHUB_REPO} /content/bomberland
%cd /content/bomberland

In [ ]:
# Cell 2 — Install deps + verify GPU
!pip install -q -r requirements.txt

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: No GPU — training will be very slow. Eval/export still work on CPU.")

In [ ]:
# Cell 3 — Mount Google Drive + symlink checkpoints/logs
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/bomberland_checkpoints
!mkdir -p /content/drive/MyDrive/bomberland_logs

!mkdir -p checkpoints logs
!ln -sf /content/drive/MyDrive/bomberland_checkpoints checkpoints/mappo
!ln -sf /content/drive/MyDrive/bomberland_logs logs/mappo

print("\n=== Checkpoints on Drive ===")
!ls -lh checkpoints/mappo/

In [ ]:
# Cell 4 — Config: best known checkpoint + training targets
import os

# Best checkpoint from eval sweep (1700 > 1800 > 1900 > latest)
BEST_CKPT = "checkpoints/mappo/checkpoint_update_1700.pth"
BACKUP_CKPT = "/content/drive/MyDrive/bomberland_checkpoints/win36_backup.pth"

# Training settings
RESUME_CKPT = BEST_CKPT          # always resume from best, NOT latest
TOTAL_STEPS = 2_500_000          # stop early if eval win drops
NUM_ENVS = 4
BASELINES = "tactical,genius,smarter"

if os.path.exists(BEST_CKPT):
    print(f"OK: {BEST_CKPT} found")
    !cp {BEST_CKPT} {BACKUP_CKPT}
    print(f"Backup saved -> {BACKUP_CKPT}")
else:
    print(f"MISSING: {BEST_CKPT}")
    print("Run BC warm-start cell below, or upload checkpoint to Drive.")

In [ ]:
# Cell 5 (optional) — BC warm-start if no checkpoint exists
import os

if not os.path.exists("checkpoints/mappo/bc_warmstart_genius.pth"):
    print("Running BC warm-start (~15 min)...")
    !python -m training.mappo.bc_warmstart \
      --teacher genius \
      --episodes 300 \
      --output checkpoints/mappo/bc_warmstart_genius.pth
else:
    print("BC warm-start already exists — skip.")

In [ ]:
# Cell 6 — Sanity tests (tracker + encoder)
!python -m pytest tests/test_encoder.py tests/test_smoke.py tests/test_reward_annealing.py -q

In [ ]:
# Cell 7 — Eval ALL checkpoints (CPU ok, ~1 min each)
import os
import subprocess

CKPT_DIR = "checkpoints/mappo"
results = []

candidates = sorted(
    f for f in os.listdir(CKPT_DIR)
    if f.endswith(".pth") and f != "bc_warmstart_genius.pth"
)

for ckpt in candidates:
    path = f"{CKPT_DIR}/{ckpt}"
    print(f"\n{'='*55}\n{ckpt}\n{'='*55}")
    subprocess.run([
        "python", "-m", "training.mappo.evaluate_agent",
        "--checkpoint", path,
        "--matches", "50",
        "--seed-suite", "0",
        "--baselines", BASELINES,
        "--device", "auto",
    ])

print("\n=== Eval history (from training) ===")
!tail -n 10 logs/mappo/eval.csv 2>/dev/null || echo "No eval.csv yet"

In [ ]:
# Cell 8 — Train from BEST checkpoint (NOT latest)
# --no-dense-anneal: keep dense reward at 1.0 (annealing hurt win rate after update 1700)
import os

resume = RESUME_CKPT
if not os.path.exists(resume):
    resume = "checkpoints/mappo/bc_warmstart_genius.pth"
    print(f"Fallback resume: {resume}")

cmd = (
    f"python -m training.mappo.train_mappo "
    f"--phase baseline "
    f"--resume {resume} "
    f"--num-envs {NUM_ENVS} "
    f"--total-steps {TOTAL_STEPS} "
    f"--no-dense-anneal"
)
print(cmd)
!{cmd}

# Watch for: dense=1.00(fixed) | trk(...) in log every 10 updates

In [ ]:
# Cell 9 — Eval after training (2 seed suites = ~100 matches)
eval_ckpt = "checkpoints/mappo/latest.pth"  # change to BEST_CKPT to re-check baseline

for suite in [0, 1]:
    print(f"\n=== seed-suite {suite} ===")
    !python -m training.mappo.evaluate_agent --checkpoint {eval_ckpt} --matches 50 --seed-suite {suite} --baselines {BASELINES} --device auto

!tail -n 5 logs/mappo/train.csv

In [ ]:
# Cell 10 (optional) — Self-play phase (run after baseline win >= 36%)
import os
selfplay_resume = BEST_CKPT if os.path.exists(BEST_CKPT) else "checkpoints/mappo/latest.pth"

!python -m training.mappo.train_mappo --phase selfplay --resume {selfplay_resume} --num-envs {NUM_ENVS} --total-steps 3000000 --no-dense-anneal

In [ ]:
# Cell 11 — Export submission from best checkpoint
# Change submit_ckpt if eval showed a different file is best
submit_ckpt = BEST_CKPT  # default: checkpoint_update_1700.pth (~36% win)

!python -m scripts.participant.export_submission --checkpoint {submit_ckpt} --output submission.zip

from google.colab import files
files.download("submission.zip")

## Quick reference

| Checkpoint | Win (re-eval) | Use for |
|---|---|---|
| `checkpoint_update_1700.pth` | **36%** | **Resume + submit** |
| `checkpoint_update_1800.pth` | 34% | — |
| `checkpoint_update_1900.pth` | 32% | — |
| `latest.pth` | 26% | Do NOT use |
| `best.pth` | 22% | Wrong metric (TrueSkill, not win rate) |

**Rules:**
- Always resume from `1700`, never `latest`
- Use `--no-dense-anneal` (annealing reduced win after update 1700)
- Stop training if eval win drops below 32%
- Backup good checkpoints to Drive manually